# Notebook 1 - Method Overview

This project is a Phi-2-only reproduction of the truth-probe generalisation experiment. The goal is not to compare many language models. The only base model used in the code and in the report is `microsoft/phi-2`.

The central question is whether a linear probe trained to distinguish true and false candidate answers on one dataset transfers to other datasets. If it transfers, that is evidence that Phi-2 has a reusable linear representation related to truth or correctness. If it does not transfer, the probe may only be learning dataset-specific artifacts.


## What This Notebook Does

This first notebook is intentionally light. It explains the data format, the prompt format, the four probe methods used in this project, and the grouped accuracy metric. It also shows a tiny dataset preview without loading Phi-2.

The heavier experiments are in the following notebooks:

1. Notebook 2 trains and compares probes on one dataset.
2. Notebook 3 tests one-to-many transfer.
3. Notebook 4 finds the best Phi-2 layer.
4. Notebook 5 builds the final transfer matrix.


In [ ]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "lie_detector_llm").exists():
            return candidate
    raise RuntimeError("Could not find the project root.")


PROJECT_ROOT = find_project_root(Path.cwd())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

RESULTS_DIR = PROJECT_ROOT / "results"
ACTIVATION_CACHE_DIR = PROJECT_ROOT / "data" / "activations"
RESULTS_DIR.mkdir(exist_ok=True)
ACTIVATION_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")


## Phi-2 Prompt Format

Phi-2 is a plain causal language model, not a chat model. Therefore the prompt should not use chat-style role markers such as `system`, `user`, or `assistant`. Every candidate answer is wrapped in the same simple completion-style template.

The probe is trained on hidden states from the final real token of this prompt. This matters because the model has seen both the question and the candidate answer at that position.


In [ ]:
from lie_detector_llm.datasets import PHI2_PROMPT_TEMPLATE
from lie_detector_llm.experiment import DEFAULT_MODEL

print("Model used in this project:", DEFAULT_MODEL)
print("
Prompt template:
")
print(PHI2_PROMPT_TEMPLATE)


## Grouped Candidate Format

Each dataset is converted into groups. A group is one question or context with several candidate answers. At least one candidate is correct and at least one is incorrect.

At evaluation time, the probe scores all candidates in the same group. The prediction is the candidate with the highest score. The group is counted as correct if the highest-scoring candidate is labelled true.

This avoids choosing a global threshold and matches the relative-ranking evaluation used in the paper.


In [ ]:
from lie_detector_llm.datasets import build_dataset_collection

preview = build_dataset_collection(
    dataset_names=["facts", "repeng_truthful"],
    max_groups=4,
    seed=0,
)

display(preview.summary())
display(preview.frame[["dataset_name", "group_id", "answer", "label"]].head(10))


## Probe Methods Used Here

The project uses the four methods that were actually run for the coursework:

| Method | Name | Uses labels? | Uses groups? | Intuition |
|---|---|---:|---:|---|
| `dim` | Difference-in-Means | yes | no | Truth direction = mean(true) - mean(false). |
| `lat` | Linear Artificial Tomography | only for orientation | no | PCA on random pairwise activation differences. |
| `lr` | Logistic Regression | yes | no | A supervised linear classifier. |
| `pca-g` | Grouped PCA | only for orientation | yes | Center each question group, then take the first principal component. |




In [ ]:
from lie_detector_llm.experiment import PROBE_METHODS

print(PROBE_METHODS)


## Interpretation

The main comparison is qualitative. The original paper found that truth probes on **Llama-2-13B** generalise best in mid-to-late layers, with a strong DIM probe trained on `dbpedia_14`. This project asks whether **Phi-2 (2.7B)** shows the same pattern despite being roughly 5x smaller.

The final answer should therefore focus on trends: early vs mid/late layers, diagonal vs off-diagonal transfer, and whether the best probe resembles the paper's best probe.